# 01 · Corpus & Pipeline Exploration

Quick EDA on the document corpus and a smoke test of the self-healing RAG graph.
Run from the project root with the dependencies in `requirements.txt` installed and
Qdrant running (`make qdrant`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
from src.config import settings
settings.llm_model, settings.embedding_model, settings.embedding_dim

## 1. Inspect the raw corpus

In [ ]:
raw_dir = Path('../data/raw')
docs = sorted(raw_dir.glob('*.txt'))
for d in docs:
    words = len(d.read_text(encoding='utf-8').split())
    print(f'{d.name:30s} {words:5d} words')

## 2. Load + chunk and look at the chunk-size distribution

In [ ]:
from src.ingestion.loader import load_and_chunk

chunks = load_and_chunk('../data/raw')
lengths = [len(c.page_content) for c in chunks]
print(f'{len(chunks)} chunks | min {min(lengths)} | max {max(lengths)} | mean {sum(lengths)//len(lengths)}')
chunks[0].metadata, chunks[0].page_content[:200]

## 3. Inspect the golden evaluation set

In [ ]:
import json
from collections import Counter

golden = json.load(open('../data/golden_dataset.json'))
print(f'{len(golden)} Q&A pairs across sources:')
Counter(g['context_source'] for g in golden)

## 4. End-to-end smoke test (requires Qdrant + API keys + an ingested index)

Run `make ingest` first, then execute the cell below.

In [ ]:
from src.graph.graph import answer_query

result = answer_query('What does the RAGAS faithfulness metric measure?')
print('Grounded:', result['grounded'], '| retries:', result['retries'])
print('Verdict :', result['critique'])
print('Answer  :', result['answer'])
print('Sources :', result['sources'])